In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
RISK_FREE = 0.045 # assumption

results_path = "backtest_results.csv"  # or full path if needed
df = pd.read_csv(results_path, parse_dates=["date"])

# Compute daily returns from equity curve
df["equity"] = df["equity"].astype(float)
df["return"] = df["equity"].pct_change().fillna(0.0)

In [ ]:
def annualized_return(returns):
    # returns daily returns series
    mean_daily = returns.mean()
    return (1 + mean_daily) ** 252 - 1

def annualized_volatility(returns):
    return returns.std() * np.sqrt(252)

def sharpe_ratio(returns, rf=RISK_FREE):
    # rf is annual risk-free, convert to daily
    daily_rf = rf / 252
    excess = returns - daily_rf
    ann_excess_ret = excess.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    return ann_excess_ret / ann_vol if ann_vol > 0 else np.nan

def sortino_ratio(returns, rf=RISK_FREE):
    daily_rf = rf / 252
    excess = returns - daily_rf
    downside = np.where(excess < 0, excess, 0)
    downside_std = np.std(downside) * np.sqrt(252)
    ann_excess_ret = excess.mean() * 252
    return ann_excess_ret / downside_std if downside_std > 0 else np.nan

def max_drawdown(equity):
    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    return drawdown.min(), drawdown


In [ ]:
ann_ret = annualized_return(df["return"])
ann_vol = annualized_volatility(df["return"])
sharpe = sharpe_ratio(df["return"])
sortino = sortino_ratio(df["return"])
mdd, dd_series = max_drawdown(df["equity"])

avg_delta = df["delta"].mean()
avg_vega = df["vega"].mean()

print("Annualized return:  {:.2%}".format(ann_ret))
print("Annualized vol:     {:.2%}".format(ann_vol))
print("Sharpe ratio:       {:.2f}".format(sharpe))
print("Sortino ratio:      {:.2f}".format(sortino))
print("Max drawdown:       {:.2%}".format(mdd))
print("Avg net delta:      {:.3f}".format(avg_delta))
print("Avg net vega:       {:.3f}".format(avg_vega))

## Visualization Below

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["date"], df["equity"], label="Equity Curve")
plt.title("Equity Curve")
plt.xlabel("Date")
plt.ylabel("Equity ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
window = 60  # 60 trading days is approximately 3 months
daily_rf = RISK_FREE / 252
excess = df["return"] - daily_rf

rolling_mean = excess.rolling(window).mean()
rolling_std = excess.rolling(window).std()
rolling_sharpe = (rolling_mean * 252) / (rolling_std * np.sqrt(252))

plt.figure(figsize=(10, 5))
plt.plot(df["date"], rolling_sharpe)
plt.title(f"{window}-Day Rolling Sharpe")
plt.xlabel("Date")
plt.ylabel("Sharpe")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
